# Step Three: SSL Experiment 2
### for Assignment 2: SSL
**By Tahiyat Chy**

This model is trained using mask autoencoding (MAE).

## Initial Setup

In [1]:
# Core imports
from pathlib import Path
import random
import time
import copy

import numpy as np
import math

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Dataset
from torchvision import datasets, transforms

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch version: 2.11.0+cu130
CUDA available: True


In [2]:
# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/gdrive')
except ImportError:
    print("Not running in Google Colab; Drive mount skipped.")

Mounted at /content/gdrive


In [3]:
# Path set to assignment directory

PROJECT_ROOT = Path('/content/gdrive/MyDrive/ITCS_6169/assignment2')
DATA_ROOT = PROJECT_ROOT / 'data'
TRAIN_ROOT = DATA_ROOT / 'train'
TEST_ROOT = DATA_ROOT / 'test'

print('Project root:', PROJECT_ROOT.resolve())
print('Training data:', TRAIN_ROOT)
print('Testing data:', TEST_ROOT)

Project root: /content/gdrive/MyDrive/ITCS_6169/assignment2
Training data: /content/gdrive/MyDrive/ITCS_6169/assignment2/data/train
Testing data: /content/gdrive/MyDrive/ITCS_6169/assignment2/data/test


In [4]:
SEED = 0

def set_random_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_random_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda


## Data Preparation
Define data augmentations. Split the training images into training and validation set.

In [5]:
import torchvision.transforms as T

BATCH_SIZE = 64
IMG_SIZE = 224
NUM_CLASSES = 16

# Data augmentation
train_transforms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandAugment(),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    T.RandomErasing(p=0.1)
])

val_transforms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load dataset
full_train_dataset = datasets.ImageFolder(root=TRAIN_ROOT)

# Split into train and val
train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
train_subset, val_subset = random_split(full_train_dataset, [train_size, val_size])

# Implement custom wrapper to apply different transforms to train and val subsets
class TransformSubset(Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.subset)

train_dataset = TransformSubset(train_subset, transform=train_transforms)
val_dataset = TransformSubset(val_subset, transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Total training images: {len(train_dataset)}")
print(f"Total validation images: {len(val_dataset)}")

Total training images: 1920
Total validation images: 480


## Model Architecture
Vision Transformer (ViT-B/16) initialized with weights=None.

In [6]:
class PatchEmbed(nn.Module):
    """2D Image to Patch Embedding"""
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=768):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)  # [B, embed_dim, H/P, W/P]
        x = x.flatten(2).transpose(1, 2)  # [B, num_patches, embed_dim]
        return x


def patchify(imgs, patch_size=16):
    """
    imgs: (N, 3, H, W)
    x: (N, L, patch_size**2 * 3)
    """
    p = patch_size
    assert imgs.shape[2] == imgs.shape[3] and imgs.shape[2] % p == 0
    h = w = imgs.shape[2] // p
    x = imgs.reshape(shape=(imgs.shape[0], 3, h, p, w, p))
    x = torch.einsum('nhpwcq->nhwpqc', x)
    x = x.reshape(shape=(imgs.shape[0], h * w, p * p * 3))
    return x


def unpatchify(x, patch_size=16):
    """
    x: (N, L, patch_size**2 * 3)
    imgs: (N, 3, H, W)
    """
    p = patch_size
    h = w = int(x.shape[1] ** 0.5)
    assert h * w == x.shape[1]
    x = x.reshape(shape=(x.shape[0], h, w, p, p, 3))
    x = torch.einsum('nhwpqc->nhpwcq', x)
    imgs = x.reshape(shape=(x.shape[0], 3, h * p, w * p))
    return imgs


class MaskedAutoencoderViT(nn.Module):
    def __init__(
        self,
        img_size=224,
        patch_size=16,
        in_chans=3,
        embed_dim=192,
        depth=12,
        num_heads=3,
        decoder_embed_dim=512,
        decoder_depth=4,
        decoder_num_heads=16,
        mlp_ratio=4.0,
        norm_pix_loss=True,
    ):
        super().__init__()

        self.patch_size = patch_size
        self.embed_dim = embed_dim
        self.norm_pix_loss = norm_pix_loss

        # --------------------------------------------------------------------------
        # MAE Encoder
        self.patch_embed = PatchEmbed(img_size, patch_size, in_chans, embed_dim)
        num_patches = self.patch_embed.num_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=int(embed_dim * mlp_ratio),
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        self.encoder_norm = nn.LayerNorm(embed_dim)
        # --------------------------------------------------------------------------

        # --------------------------------------------------------------------------
        # MAE Decoder
        self.decoder_embed = nn.Linear(embed_dim, decoder_embed_dim, bias=True)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, decoder_embed_dim))
        self.decoder_pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, decoder_embed_dim))

        decoder_layer = nn.TransformerEncoderLayer(
            d_model=decoder_embed_dim,
            nhead=decoder_num_heads,
            dim_feedforward=int(decoder_embed_dim * mlp_ratio),
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.decoder = nn.TransformerEncoder(decoder_layer, num_layers=decoder_depth)
        self.decoder_norm = nn.LayerNorm(decoder_embed_dim)
        self.decoder_pred = nn.Linear(decoder_embed_dim, patch_size**2 * in_chans, bias=True)
        # --------------------------------------------------------------------------

        self._initialize_weights()

    def _initialize_weights(self):
        torch.nn.init.trunc_normal_(self.pos_embed, std=0.02)
        torch.nn.init.trunc_normal_(self.decoder_pos_embed, std=0.02)
        torch.nn.init.normal_(self.cls_token, std=0.02)
        torch.nn.init.normal_(self.mask_token, std=0.02)

    def random_masking(self, x, mask_ratio=0.75):
        """
        Perform per-sample random masking by shuffling patch indices.
        x: [N, L, D], sequence
        """
        N, L, D = x.shape
        len_keep = int(L * (1 - mask_ratio))

        noise = torch.rand(N, L, device=x.device)  # noise in [0, 1]

        # Sort noise for each sample
        ids_shuffle = torch.argsort(noise, dim=1)  # ascend: small is keep, large is remove
        ids_restore = torch.argsort(ids_shuffle, dim=1)

        # Keep first subset
        ids_keep = ids_shuffle[:, :len_keep]
        x_masked = torch.gather(x, dim=1, index=ids_keep.unsqueeze(-1).repeat(1, 1, D))

        # Generate binary mask: 0 is keep, 1 is remove
        mask = torch.ones([N, L], device=x.device)
        mask[:, :len_keep] = 0
        mask = torch.gather(mask, dim=1, index=ids_restore)

        return x_masked, mask, ids_restore

    def forward_encoder(self, x, mask_ratio=0.75):
        x = self.patch_embed(x)
        x = x + self.pos_embed[:, 1:, :]

        # Masking: [N, L, D] -> [N, L*(1-mask_ratio), D]
        x, mask, ids_restore = self.random_masking(x, mask_ratio)

        # Append CLS token
        cls_token = self.cls_token + self.pos_embed[:, :1, :]
        cls_tokens = cls_token.expand(x.shape[0], -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)

        # Apply Transformer blocks
        x = self.encoder(x)
        x = self.encoder_norm(x)

        return x, mask, ids_restore

    def forward_decoder(self, x, ids_restore):
        x = self.decoder_embed(x)

        # Append mask tokens to sequence
        mask_tokens = self.mask_token.repeat(x.shape[0], ids_restore.shape[1] + 1 - x.shape[1], 1)
        x_ = torch.cat([x[:, 1:, :], mask_tokens], dim=1)  # exclude CLS
        x_ = torch.gather(x_, dim=1, index=ids_restore.unsqueeze(-1).repeat(1, 1, x.shape[2]))  # unshuffle
        x = torch.cat([x[:, :1, :], x_], dim=1)  # append CLS back

        x = x + self.decoder_pos_embed

        x = self.decoder(x)
        x = self.decoder_norm(x)

        # Predict patch pixels
        x = self.decoder_pred(x)
        x = x[:, 1:, :]  # remove CLS token

        return x

    def forward_loss(self, imgs, pred, mask):
        target = patchify(imgs, self.patch_size)
        if self.norm_pix_loss:
            mean = target.mean(dim=-1, keepdim=True)
            var = target.var(dim=-1, keepdim=True)
            target = (target - mean) / (var + 1e-6) ** 0.5

        loss = (pred - target) ** 2
        loss = loss.mean(dim=-1)  # [N, L], loss per patch

        loss = (loss * mask).sum() / mask.sum()  # Mean loss over masked patches only
        return loss

    def forward(self, imgs, mask_ratio=0.75):
        latent, mask, ids_restore = self.forward_encoder(imgs, mask_ratio)
        pred = self.forward_decoder(latent, ids_restore)
        loss = self.forward_loss(imgs, pred, mask)
        return loss, pred, mask

## Training Loop
Using CrossEntropyLoss and AdamW optimizer.

In [8]:
def pretrain_mae(model, train_loader, epochs=300, lr=1e-3, mask_ratio=0.75):
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05, betas=(0.9, 0.95))
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    model.train()
    print("--- Starting MAE Pre-training ---")
    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, _ in train_loader:  # Labels are ignored
            inputs = inputs.to(device)

            optimizer.zero_grad()
            loss, _, _ = model(inputs, mask_ratio=mask_ratio)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)

        scheduler.step()
        epoch_loss = running_loss / len(train_loader.dataset)
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Pre-train Epoch [{epoch+1}/{epochs}] | Reconstruct Loss: {epoch_loss:.5f}")

    print("MAE Pre-training Complete.\n")
    return model


# Instantiate and pre-train MAE
mae_model = MaskedAutoencoderViT(img_size=IMG_SIZE, patch_size=16).to(device)
mae_model = pretrain_mae(mae_model, train_loader, epochs=300, lr=1e-3, mask_ratio=0.75)

--- Starting MAE Pre-training ---


/tmp/ipykernel_7379/1909665080.py:81: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=depth)
/tmp/ipykernel_7379/1909665080.py:99: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.decoder = nn.TransformerEncoder(decoder_layer, num_layers=decoder_depth)


Pre-train Epoch [1/300] | Reconstruct Loss: 1.04479
Pre-train Epoch [5/300] | Reconstruct Loss: 0.97542
Pre-train Epoch [10/300] | Reconstruct Loss: 0.95178
Pre-train Epoch [15/300] | Reconstruct Loss: 0.91585
Pre-train Epoch [20/300] | Reconstruct Loss: 0.89724
Pre-train Epoch [25/300] | Reconstruct Loss: 0.88562
Pre-train Epoch [30/300] | Reconstruct Loss: 0.88883
Pre-train Epoch [35/300] | Reconstruct Loss: 0.86894
Pre-train Epoch [40/300] | Reconstruct Loss: 0.87095
Pre-train Epoch [45/300] | Reconstruct Loss: 0.85141
Pre-train Epoch [50/300] | Reconstruct Loss: 0.84116
Pre-train Epoch [55/300] | Reconstruct Loss: 0.83646
Pre-train Epoch [60/300] | Reconstruct Loss: 0.80459
Pre-train Epoch [65/300] | Reconstruct Loss: 0.72577
Pre-train Epoch [70/300] | Reconstruct Loss: 0.66467
Pre-train Epoch [75/300] | Reconstruct Loss: 0.62107
Pre-train Epoch [80/300] | Reconstruct Loss: 0.60585
Pre-train Epoch [85/300] | Reconstruct Loss: 0.58848
Pre-train Epoch [90/300] | Reconstruct Loss: 0.5

## Model Fine-Tuning

In [9]:
class ViTForClassification(nn.Module):
    def __init__(self, mae_model, num_classes=16):
        super().__init__()
        # Share or copy pre-trained encoder modules
        self.patch_embed = mae_model.patch_embed
        self.cls_token = mae_model.cls_token
        self.pos_embed = mae_model.pos_embed
        self.encoder = mae_model.encoder
        self.encoder_norm = mae_model.encoder_norm

        # Classification head
        self.head = nn.Linear(mae_model.embed_dim, num_classes)

    def forward(self, x):
        x = self.patch_embed(x)
        x = x + self.pos_embed[:, 1:, :]

        cls_token = self.cls_token + self.pos_embed[:, :1, :]
        cls_tokens = cls_token.expand(x.shape[0], -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)

        x = self.encoder(x)
        x = self.encoder_norm(x)

        # Extract [CLS] token output
        cls_out = x[:, 0]
        return self.head(cls_out)


# Initialize downstream classifier with pre-trained encoder weights
finetune_model = ViTForClassification(mae_model, num_classes=NUM_CLASSES).to(device)

In [10]:
# --------------------------------------------------------------------------
# Helper functions for training and evaluation
# --------------------------------------------------------------------------
def train_one_epoch(model, dataloader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
    return running_loss / len(dataloader.dataset)


def evaluate(model, dataloader):
    model.eval()
    corrects = 0
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            corrects += torch.sum(preds == labels.data)
    return corrects.double() / len(dataloader.dataset)

In [12]:
def train_finetune(model, train_loader, val_loader, lr=1e-4):
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    # ==========================================================================
    # STAGE 1: Linear Probing (Freeze Backbone, Train Classification Head Only)
    # ==========================================================================
    print("--- Starting Stage 1: Linear Probing Head ---")

    # Freeze encoder and patch embedding parameters
    for param in finetune_model.encoder.parameters():
        param.requires_grad = False
    for param in finetune_model.patch_embed.parameters():
        param.requires_grad = False
    finetune_model.pos_embed.requires_grad = False
    finetune_model.cls_token.requires_grad = False

    # Optimizer configured strictly for the output head
    head_optimizer = optim.AdamW(
        finetune_model.head.parameters(), lr=1e-3, weight_decay=0.05
    )

    PROBE_EPOCHS = 10
    for epoch in range(PROBE_EPOCHS):
        train_loss = train_one_epoch(
            finetune_model, train_loader, criterion, head_optimizer
        )
        val_acc = evaluate(finetune_model, val_loader)
        print(
            f"Probe Epoch [{epoch+1}/{PROBE_EPOCHS}] | Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f}"
        )

    # ==========================================================================
    # STAGE 2: Full Fine-Tuning (Unfreeze All Layers, Low Learning Rate)
    # ==========================================================================
    print("\n--- Starting Stage 2: Full Model Fine-Tuning ---")

    # Unfreeze all parameters across the network
    for param in finetune_model.parameters():
        param.requires_grad = True

    # Optimizer for the full network with a lower learning rate to preserve features
    ft_optimizer = optim.AdamW(
        finetune_model.parameters(), lr=3e-5, weight_decay=0.05
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        ft_optimizer, mode="max", factor=0.5, patience=3
    )

    best_acc = 0.0
    best_wts = copy.deepcopy(finetune_model.state_dict())

    FT_EPOCHS = 20
    for epoch in range(FT_EPOCHS):
        train_loss = train_one_epoch(
            finetune_model, train_loader, criterion, ft_optimizer
        )
        val_acc = evaluate(finetune_model, val_loader)
        scheduler.step(val_acc)

        print(
            f"FT Epoch [{epoch+1}/{FT_EPOCHS}] | Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f}"
        )

        if val_acc > best_acc:
            best_acc = val_acc
            best_wts = copy.deepcopy(finetune_model.state_dict())

    print(f"\nTraining Complete. Best Validation Accuracy: {best_acc:.4f}")
    finetune_model.load_state_dict(best_wts)

# Run downstream fine-tuning
final_mae_model = train_finetune(finetune_model, train_loader, val_loader, lr=1e-4)

--- Starting Stage 1: Linear Probing Head ---
Probe Epoch [1/10] | Loss: 2.7753 | Val Acc: 0.1104
Probe Epoch [2/10] | Loss: 2.7422 | Val Acc: 0.1854
Probe Epoch [3/10] | Loss: 2.7222 | Val Acc: 0.1875
Probe Epoch [4/10] | Loss: 2.7027 | Val Acc: 0.2542
Probe Epoch [5/10] | Loss: 2.6832 | Val Acc: 0.2417
Probe Epoch [6/10] | Loss: 2.6636 | Val Acc: 0.2583
Probe Epoch [7/10] | Loss: 2.6462 | Val Acc: 0.2958
Probe Epoch [8/10] | Loss: 2.6305 | Val Acc: 0.2896
Probe Epoch [9/10] | Loss: 2.6171 | Val Acc: 0.3000
Probe Epoch [10/10] | Loss: 2.6011 | Val Acc: 0.2875

--- Starting Stage 2: Full Model Fine-Tuning ---
FT Epoch [1/20] | Loss: 2.5081 | Val Acc: 0.2208
FT Epoch [2/20] | Loss: 2.3761 | Val Acc: 0.2750
FT Epoch [3/20] | Loss: 2.2998 | Val Acc: 0.3187
FT Epoch [4/20] | Loss: 2.2358 | Val Acc: 0.3646
FT Epoch [5/20] | Loss: 2.1841 | Val Acc: 0.3771
FT Epoch [6/20] | Loss: 2.1238 | Val Acc: 0.3875
FT Epoch [7/20] | Loss: 2.0721 | Val Acc: 0.4146
FT Epoch [8/20] | Loss: 2.0526 | Val Acc

In [13]:
# EXPERIMENT: Put fine-tuned model through the fine-tuning pipeline a second time
final_mae_model = train_finetune(final_mae_model, train_loader, val_loader, lr=1e-4)

--- Starting Stage 1: Linear Probing Head ---
Probe Epoch [1/10] | Loss: 1.7703 | Val Acc: 0.5479
Probe Epoch [2/10] | Loss: 1.7395 | Val Acc: 0.5563
Probe Epoch [3/10] | Loss: 1.7292 | Val Acc: 0.5625
Probe Epoch [4/10] | Loss: 1.7170 | Val Acc: 0.5604
Probe Epoch [5/10] | Loss: 1.7183 | Val Acc: 0.5625
Probe Epoch [6/10] | Loss: 1.7060 | Val Acc: 0.5729
Probe Epoch [7/10] | Loss: 1.6846 | Val Acc: 0.5646
Probe Epoch [8/10] | Loss: 1.6997 | Val Acc: 0.5708
Probe Epoch [9/10] | Loss: 1.6680 | Val Acc: 0.5646
Probe Epoch [10/10] | Loss: 1.6890 | Val Acc: 0.5792

--- Starting Stage 2: Full Model Fine-Tuning ---
FT Epoch [1/20] | Loss: 1.6717 | Val Acc: 0.5646
FT Epoch [2/20] | Loss: 1.6725 | Val Acc: 0.5583
FT Epoch [3/20] | Loss: 1.6725 | Val Acc: 0.5896
FT Epoch [4/20] | Loss: 1.6594 | Val Acc: 0.5875
FT Epoch [5/20] | Loss: 1.6268 | Val Acc: 0.5833
FT Epoch [6/20] | Loss: 1.6255 | Val Acc: 0.5896
FT Epoch [7/20] | Loss: 1.6079 | Val Acc: 0.5813
FT Epoch [8/20] | Loss: 1.5877 | Val Acc

In [14]:
# EXPERIMENT: Put the pre-trained model (before any previous fine-tuning) through an extended fine-tuning pipeline
# ==========================================================================
# EXTENDED FINETUNING
# ==========================================================================

def train_finetune_ext(model, train_loader, val_loader, lr=1e-4):
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    # ==========================================================================
    # STAGE 1: Linear Probing (Freeze Backbone, Train Classification Head Only)
    # ==========================================================================
    print("--- Starting Stage 1: Linear Probing Head ---")

    # Freeze encoder and patch embedding parameters
    for param in finetune_model.encoder.parameters():
        param.requires_grad = False
    for param in finetune_model.patch_embed.parameters():
        param.requires_grad = False
    finetune_model.pos_embed.requires_grad = False
    finetune_model.cls_token.requires_grad = False

    # Optimizer configured strictly for the output head
    head_optimizer = optim.AdamW(
        finetune_model.head.parameters(), lr=1e-3, weight_decay=0.05
    )

    PROBE_EPOCHS = 15
    for epoch in range(PROBE_EPOCHS):
        train_loss = train_one_epoch(
            finetune_model, train_loader, criterion, head_optimizer
        )
        val_acc = evaluate(finetune_model, val_loader)
        print(
            f"Probe Epoch [{epoch+1}/{PROBE_EPOCHS}] | Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f}"
        )

    # ==========================================================================
    # STAGE 2: Full Fine-Tuning (Unfreeze All Layers, Low Learning Rate)
    # ==========================================================================
    print("\n--- Starting Stage 2: Full Model Fine-Tuning ---")

    # Unfreeze all parameters across the network
    for param in finetune_model.parameters():
        param.requires_grad = True

    # Optimizer for the full network with a lower learning rate to preserve features
    ft_optimizer = optim.AdamW(
        finetune_model.parameters(), lr=3e-5, weight_decay=0.05
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        ft_optimizer, mode="max", factor=0.5, patience=3
    )

    best_acc = 0.0
    best_wts = copy.deepcopy(finetune_model.state_dict())

    FT_EPOCHS = 100
    for epoch in range(FT_EPOCHS):
        train_loss = train_one_epoch(
            finetune_model, train_loader, criterion, ft_optimizer
        )
        val_acc = evaluate(finetune_model, val_loader)
        scheduler.step(val_acc)

        print(
            f"FT Epoch [{epoch+1}/{FT_EPOCHS}] | Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f}"
        )

        if val_acc > best_acc:
            best_acc = val_acc
            best_wts = copy.deepcopy(finetune_model.state_dict())

    print(f"\nTraining Complete. Best Validation Accuracy: {best_acc:.4f}")
    finetune_model.load_state_dict(best_wts)

# Run downstream fine-tuning
final_mae_model = train_finetune_ext(finetune_model, train_loader, val_loader, lr=1e-4)

--- Starting Stage 1: Linear Probing Head ---
Probe Epoch [1/15] | Loss: 1.6013 | Val Acc: 0.5896
Probe Epoch [2/15] | Loss: 1.5800 | Val Acc: 0.5958
Probe Epoch [3/15] | Loss: 1.5612 | Val Acc: 0.6104
Probe Epoch [4/15] | Loss: 1.5579 | Val Acc: 0.6000
Probe Epoch [5/15] | Loss: 1.5705 | Val Acc: 0.5938
Probe Epoch [6/15] | Loss: 1.5421 | Val Acc: 0.5979
Probe Epoch [7/15] | Loss: 1.5519 | Val Acc: 0.6000
Probe Epoch [8/15] | Loss: 1.5668 | Val Acc: 0.5958
Probe Epoch [9/15] | Loss: 1.5797 | Val Acc: 0.5979
Probe Epoch [10/15] | Loss: 1.5545 | Val Acc: 0.6042
Probe Epoch [11/15] | Loss: 1.5662 | Val Acc: 0.5917
Probe Epoch [12/15] | Loss: 1.5566 | Val Acc: 0.6083
Probe Epoch [13/15] | Loss: 1.5386 | Val Acc: 0.6167
Probe Epoch [14/15] | Loss: 1.5518 | Val Acc: 0.6042
Probe Epoch [15/15] | Loss: 1.5556 | Val Acc: 0.6021

--- Starting Stage 2: Full Model Fine-Tuning ---
FT Epoch [1/100] | Loss: 1.5518 | Val Acc: 0.6146
FT Epoch [2/100] | Loss: 1.5323 | Val Acc: 0.6208
FT Epoch [3/100] |